<a href="https://colab.research.google.com/github/r4k0nb4k0n/browser-engineering-study/blob/main/part-1-loading-pages.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1부 페이지 로딩

## 1장 웹페이지 다운로드

### 1.1 서버에 연결하기



In [ ]:
!apt-get update
!apt-get install netcat

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:4 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:6 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:8 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [99.9 kB]
Get:9 https://cli.github.com/packages stable/main amd64 Packages [355 B]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [3,105 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 Packages [4,424 kB]
Get:14 https://r2u

In [ ]:
import subprocess

try:
    # Use 'nc -z' for zero-I/O mode to check port reachability and immediately close.
    # Added a small timeout for subprocess.run as a failsafe.
    result = subprocess.run(
        ["nc", "-vz", "google.com", "80"],
        capture_output=True,
        text=True,
        timeout=5 # Set a timeout of 5 seconds as a failsafe
    )
    print("Standard Output:")
    print(result.stdout)
    print("Standard Error:")
    print(result.stderr)
except subprocess.TimeoutExpired:
    print("Error: The 'nc -z' command timed out after 5 seconds. This might indicate network issues or that the host is unreachable.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")


Standard Output:

Standard Error:
Connection to google.com (173.194.217.101) 80 port [tcp/*] succeeded!



운영체제가 `google.com` 이라는 호스트 이름을 `142.251.2.113`이라는 IP 주소로 변환했고 그 주소에 연결했습니다. 여러분은 이제 `google.com`과 메시지를 주고받을 수 있습니다.

### 1.2 정보 요청하기

In [ ]:
import subprocess

try:
    # Define the 'nc' command arguments as a list
    command_args = ["nc", "-v", "google.com", "80"]

    # Define the multiline input data
    input_data = """
GET / HTTP/1.0
Host: google.com

"""

    # Use subprocess.run, passing the input via the 'input' argument
    # No need for shell=True when passing input this way
    result = subprocess.run(
        command_args,
        capture_output=True,
        text=True,
        input=input_data, # Provide multiline input directly
        timeout=5 # Set a timeout of 5 seconds as a failsafe
    )
    print("Standard Output:")
    print(result.stdout)
    print("Standard Error:")
    print(result.stderr)
except subprocess.TimeoutExpired:
    print("Error: The 'nc' command timed out after 5 seconds. This might indicate network issues or that the host is unreachable.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

Standard Output:
HTTP/1.0 301 Moved Permanently
Location: http://www.google.com/
Content-Type: text/html; charset=UTF-8
Content-Security-Policy-Report-Only: object-src 'none';base-uri 'self';script-src 'nonce-zuK_qoqLsmN-XEg8xAnzSQ' 'strict-dynamic' 'report-sample' 'unsafe-eval' 'unsafe-inline' https: http:;report-uri https://csp.withgoogle.com/csp/gws/other-hp
Date: Sun, 12 Jul 2026 13:34:59 GMT
Expires: Tue, 11 Aug 2026 13:34:59 GMT
Cache-Control: public, max-age=2592000
Server: gws
Content-Length: 219
X-XSS-Protection: 0
X-Frame-Options: SAMEORIGIN

<HTML><HEAD><meta http-equiv="content-type" content="text/html;charset=utf-8">
<TITLE>301 Moved</TITLE></HEAD><BODY>
<H1>301 Moved</H1>
The document has moved
<A HREF="http://www.google.com/">here</A>.
</BODY></HTML>

Standard Error:
Connection to google.com (173.194.212.102) 80 port [tcp/*] succeeded!



### 1.3 서버의 응답

https://datatracker.ietf.org/doc/html/rfc1945#section-6.1

> The first line of a Full-Response message is the Status-Line,
>consisting of the protocol version followed by a numeric status code
>and its associated textual phrase, with each element separated by SP
>characters. No CR or LF is allowed except in the final CRLF sequence.
>
> Status-Line = HTTP-Version SP Status-Code SP Reason-Phrase CRLF
>
>Since a status line always begins with the protocol version and
>status code
>
> "HTTP/" 1*DIGIT "." 1*DIGIT SP 3DIGIT SP
>
>(e.g., "HTTP/1.0 200 "), the presence of that expression is
>sufficient to differentiate a Full-Response from a Simple-Response.
>Although the Simple-Response format may allow such an expression to
>occur at the beginning of an entity body, and thus cause a
>misinterpretation of the message if it was given in response to a
>Full-Request, most HTTP/0.9 servers are limited to responses of type
>"text/html" and therefore would never generate such a response.

https://datatracker.ietf.org/doc/html/rfc1945#section-6.2

>The response header fields allow the server to pass additional
>information about the response which cannot be placed in the Status-
>Line. These header fields give information about the server and about
>further access to the resource identified by the Request-URI.
>
>    Response-Header = Location                ; Section 10.11
>                    | Server                  ; Section 10.14
>                    | WWW-Authenticate        ; Section 10.16
>
>Response-Header field names can be extended reliably only in
>combination with a change in the protocol version. However, new or
>experimental header fields may be given the semantics of response
>header fields if all parties in the communication recognize them to
> be response header fields. Unrecognized header fields are treated as
>Entity-Header fields.

### 1.4 파이썬을 통한 텔넷

### 1.5 요청과 응답

### 1.6 HTML 표시하기

In [4]:
import socket

class URL:
  def __init__(self, url):
    self.scheme, url = url.split("://", 1)
    assert self.scheme == "http"
    # ...
    if "/" not in url:
      url = url + "/"
    self.host, url = url.split("/", 1)
    self.path = "/" + url

  def request(self):
    s = socket.socket(
        family=socket.AF_INET,
        type=socket.SOCK_STREAM,
        proto=socket.IPPROTO_TCP
    )
    s.connect((self.host, 80))
    request = "GET {} HTTP/1.0\r\n".format(self.path)
    request += "Host: {}\r\n".format(self.host)
    request += "\r\n" # 이걸 안하면 서로 데드락이 걸릴 것.
    s.send(request.encode("utf8"))
    response = s.makefile("r", encoding="utf8", newline="\r\n")
    statusline = response.readline()
    version, status, explanation = statusline.split(" ", 2)
    response_headers = {}
    while True:
      line = response.readline()
      if line == "\r\n": break
      header, value = line.split(":", 1)
      response_headers[header.casefold()] = value.strip()
    assert "transfer-encoding" not in response_headers
    assert "content-encoding" not in response_headers
    body = response.read()
    s.close()
    return body

def show(body):
  in_tag = False
  for c in body:
    if c == "<":
      in_tag = True
    elif c == ">":
      in_tag = False
    elif not in_tag:
      print(c, end="")

def load(url):
  body = url.request()
  show(body)

load(URL("http://google.com"))


301 Moved
301 Moved
The document has moved
here.



### 1.7 암호화된 연결

In [6]:
import socket
import ssl

class URL:
  def __init__(self, url):
    self.scheme, url = url.split("://", 1)
    assert self.scheme in ["http", "https"]
    # ...
    if "/" not in url:
      url = url + "/"
    self.host, url = url.split("/", 1)
    self.path = "/" + url

  def request(self):
    s = socket.socket(
        family=socket.AF_INET,
        type=socket.SOCK_STREAM,
        proto=socket.IPPROTO_TCP
    )
    if self.scheme == "http":
      self.port = 80
    elif self.scheme == "https":
      self.port = 443
    if self.scheme == "https":
      ctx = ssl.create_default_context()
      s = ctx.wrap_socket(s, server_hostname=self.host)
    if ":" in self.host:
      self.host, port = self.host.split(":", 1)
      self.port = int(port)
    s.connect((self.host, self.port))
    request = "GET {} HTTP/1.0\r\n".format(self.path)
    request += "Host: {}\r\n".format(self.host)
    request += "\r\n" # 이걸 안하면 서로 데드락이 걸릴 것.
    s.send(request.encode("utf8"))
    response = s.makefile("r", encoding="utf8", newline="\r\n")
    statusline = response.readline()
    version, status, explanation = statusline.split(" ", 2)
    response_headers = {}
    while True:
      line = response.readline()
      if line == "\r\n": break
      header, value = line.split(":", 1)
      response_headers[header.casefold()] = value.strip()
    assert "transfer-encoding" not in response_headers
    assert "content-encoding" not in response_headers
    body = response.read()
    s.close()
    return body

def show(body):
  in_tag = False
  for c in body:
    if c == "<":
      in_tag = True
    elif c == ">":
      in_tag = False
    elif not in_tag:
      print(c, end="")

def load(url):
  body = url.request()
  show(body)

load(URL("https://browser.engineering/examples/example1-simple.html"))


  
    This is a simple
    web page with some
    text in it.
  



### 1.8 요약

- URL을 스킴, 호스트, 포트, 경로로 파싱합니다.
- `sockets`와 `ssl` 라이브러리를 사용해 해당 호스트에 연결합니다.
- `Host` 헤더를 포함해 해당 호스트에 연결합니다.
- HTTP 응답을 상태 표시줄, 헤더, 바디로 분할합니다.
- 바디에서 태그가 아닌 텍스트를 출력합니다.

### 1.9 연습문제

#### 1-1 HTTP/1.1

In [16]:
import socket
import ssl

class URL:
  def __init__(self, url):
    self.scheme, url = url.split("://", 1)
    assert self.scheme in ["http", "https"]
    # ...
    if "/" not in url:
      url = url + "/"
    self.host, url = url.split("/", 1)
    self.path = "/" + url

  def request(self, headers=None):
    s = socket.socket(
        family=socket.AF_INET,
        type=socket.SOCK_STREAM,
        proto=socket.IPPROTO_TCP
    )
    if self.scheme == "http":
      self.port = 80
    elif self.scheme == "https":
      self.port = 443
    if self.scheme == "https":
      ctx = ssl.create_default_context()
      s = ctx.wrap_socket(s, server_hostname=self.host)
    if ":" in self.host:
      self.host, port = self.host.split(":", 1)
      self.port = int(port)
    s.connect((self.host, self.port))

    default_headers = {
        "Host": self.host,
        "Connection": "close",
        "User-Agent": "Shenanigan"
    }
    if headers:
        default_headers.update(headers)
    request_lines = [f"GET {self.path} HTTP/1.1\r\n"]
    for header_name, header_value in default_headers.items():
        request_lines.append(f"{header_name}: {header_value}\r\n")
    request_lines.append("\r\n") # 이걸 안하면 서로 데드락이 걸릴 것.

    request_data = "".join(request_lines).encode("utf8")
    s.send(request_data)

    response = s.makefile("r", encoding="utf8", newline="\r\n")
    statusline = response.readline()
    version, status, explanation = statusline.split(" ", 2)
    response_headers = {}
    while True:
      line = response.readline()
      if line == "\r\n": break
      header, value = line.split(":", 1)
      response_headers[header.casefold()] = value.strip()
    assert "transfer-encoding" not in response_headers
    assert "content-encoding" not in response_headers
    body = response.read()
    s.close()
    return body

def show(body):
  in_tag = False
  for c in body:
    if c == "<":
      in_tag = True
    elif c == ">":
      in_tag = False
    elif not in_tag:
      print(c, end="")

def load(url, headers=None):
  body = url.request(headers=headers)
  show(body)


load(URL("http://httpbin.org/get"), {"Latte": "is horse."})

{
  "args": {}, 
  "headers": {
    "Host": "httpbin.org", 
    "Latte": "is horse.", 
    "User-Agent": "Shenanigan", 
    "X-Amzn-Trace-Id": "Root=1-6a54c0ae-3a372c6176c176bc1906a1e6"
  }, 
  "origin": "34.81.86.44", 
  "url": "http://httpbin.org/get"
}


#### 1-2 파일 URL

In [22]:
import socket
import ssl
import pathlib

class URL:
  def __init__(self, url):
    self.scheme, url = url.split("://", 1)
    assert self.scheme in ["http", "https", "file"]
    # ...
    if "/" not in url:
      url = url + "/"
    self.host, url = url.split("/", 1)
    self.path = "/" + url

  def request(self, headers=None):
    if self.scheme == "file":
      # For file scheme, read directly from the local file system
      # The path will be like '/mnt' or '/content/sample_data/README.md'
      # The self.path already includes the leading '/', so use it directly for absolute path
      file_path = self.path # Changed to use self.path directly
      with open(file_path, "r") as f:
        return f.read()

    s = socket.socket(
        family=socket.AF_INET,
        type=socket.SOCK_STREAM,
        proto=socket.IPPROTO_TCP
    )
    if self.scheme == "http":
      self.port = 80
    elif self.scheme == "https":
      self.port = 443
    if self.scheme == "https":
      ctx = ssl.create_default_context()
      s = ctx.wrap_socket(s, server_hostname=self.host)
    if ":" in self.host:
      self.host, port = self.host.split(":", 1)
      self.port = int(port)
    s.connect((self.host, self.port))

    default_headers = {
        "Host": self.host,
        "Connection": "close",
        "User-Agent": "Shenanigan"
    }
    if headers:
        default_headers.update(headers)
    request_lines = [f"GET {self.path} HTTP/1.1\r\n"]
    for header_name, header_value in default_headers.items():
        request_lines.append(f"{header_name}: {header_value}\r\n")
    request_lines.append("\r\n") # 이걸 안하면 서로 데드락이 걸릴 것.

    request_data = "".join(request_lines).encode("utf8")
    s.send(request_data)

    response = s.makefile("r", encoding="utf8", newline="\r\n")
    statusline = response.readline()
    version, status, explanation = statusline.split(" ", 2)
    response_headers = {}
    while True:
      line = response.readline()
      if line == "\r\n": break
      header, value = line.split(":", 1)
      response_headers[header.casefold()] = value.strip()
    assert "transfer-encoding" not in response_headers
    assert "content-encoding" not in response_headers
    body = response.read()
    s.close()
    return body

def show(body):
  in_tag = False
  for c in body:
    if c == "<":
      in_tag = True
    elif c == ">":
      in_tag = False
    elif not in_tag:
      print(c, end="")

def load(url, headers=None):
  body = url.request(headers=headers) # Pass headers correctly
  show(body)


# Updated to use a readable file in the Colab environment
load(URL("file:///content/sample_data/README.md"))

This directory includes a few sample datasets to get you started.

*   `california_housing_data*.csv` is California housing data from the 1990 US
    Census; more information is available at:
    https://docs.google.com/document/d/e/2PACX-1vRhYtsvc5eOR2FWNCwaBiKL6suIOrxJig8LcSBbmCbyYsayia_DvPOOBlXZ4CAlQ5nlDD8kTaIDRwrN/pub

*   `mnist_*.csv` is a small sample of the
    [MNIST database](https://en.wikipedia.org/wiki/MNIST_database), which is
    described at: http://yann.lecun.com/exdb/mnist/

*   `anscombe.json` contains a copy of
    [Anscombe's quartet](https://en.wikipedia.org/wiki/Anscombe%27s_quartet); it
    was originally described in

    Anscombe, F. J. (1973). 'Graphs in Statistical Analysis'. American
    Statistician. 27 (1): 17-21. JSTOR 2682899.

    and our copy was prepared by the
    [vega_datasets library](https://github.com/altair-viz/vega_datasets/blob/4f67bdaad10f45e3549984e17e1b3088c731503d/vega_datasets/_data/anscombe.json).


#### 1-3 data: 스킴

In [24]:
import socket
import ssl
import pathlib

class URL:
  def __init__(self, url):
    # split by the first ':' to get the scheme
    self.scheme, remainder = url.split(":", 1)
    assert self.scheme in ["http", "https", "file", "data"]

    self.host = None
    self.path = None
    self.data = None

    if self.scheme == "data":
      # data scheme: remainder looks like 'text/html,Hello'
      if "," not in remainder:
          raise ValueError(f"Invalid data URL: {url} - missing comma.")
      _, data_content = remainder.split(",", 1)
      self.data = data_content
    else:
      # for http, https, file, skip the '//' if present
      if remainder.startswith("//"):
        remainder = remainder[2:]

      if "/" not in remainder:
        remainder = remainder + "/"
      self.host, remainder = remainder.split("/", 1)
      self.path = "/" + remainder

  def request(self, headers=None):
    if self.scheme == "data":
      return self.data

    if self.scheme == "file":
      file_path = self.path
      with open(file_path, "r") as f:
        return f.read()

    s = socket.socket(
        family=socket.AF_INET,
        type=socket.SOCK_STREAM,
        proto=socket.IPPROTO_TCP
    )
    if self.scheme == "http":
      self.port = 80
    elif self.scheme == "https":
      self.port = 443
    if self.scheme == "https":
      ctx = ssl.create_default_context()
      s = ctx.wrap_socket(s, server_hostname=self.host)
    if ":" in self.host:
      self.host, port = self.host.split(":", 1)
      self.port = int(port)
    s.connect((self.host, self.port))

    default_headers = {
        "Host": self.host,
        "Connection": "close",
        "User-Agent": "Shenanigan"
    }
    if headers:
        default_headers.update(headers)
    request_lines = [f"GET {self.path} HTTP/1.1\r\n"]
    for header_name, header_value in default_headers.items():
        request_lines.append(f"{header_name}: {header_value}\r\n")
    request_lines.append("\r\n")

    request_data = "".join(request_lines).encode("utf8")
    s.send(request_data)

    response = s.makefile("r", encoding="utf8", newline="\r\n")
    statusline = response.readline()
    version, status, explanation = statusline.split(" ", 2)
    response_headers = {}
    while True:
      line = response.readline()
      if line == "\r\n": break
      header, value = line.split(":", 1)
      response_headers[header.casefold()] = value.strip()
    assert "transfer-encoding" not in response_headers
    assert "content-encoding" not in response_headers
    body = response.read()
    s.close()
    return body

def show(body):
  in_tag = False
  for c in body:
    if c == "<":
      in_tag = True
    elif c == ">":
      in_tag = False
    elif not in_tag:
      print(c, end="")

def load(url, headers=None):
  body = url.request(headers=headers)
  show(body)

# Test with data scheme (using standard data: format)
load(URL("data:text/html,<h1>Hello from Data Scheme!</h1><p>This is a test.</p>"))

Hello from Data Scheme!This is a test.

#### 1-4 HTML 엔티티(특수코드)

In [27]:
import socket
import ssl
import pathlib

class URL:
  def __init__(self, url):
    # split by the first ':' to get the scheme
    self.scheme, remainder = url.split(":", 1)
    assert self.scheme in ["http", "https", "file", "data"]

    self.host = None
    self.path = None
    self.data = None

    if self.scheme == "data":
      # data scheme: remainder looks like 'text/html,Hello'
      if "," not in remainder:
          raise ValueError(f"Invalid data URL: {url} - missing comma.")
      _, data_content = remainder.split(",", 1)
      self.data = data_content
    else:
      # for http, https, file, skip the '//' if present
      if remainder.startswith("//"):
        remainder = remainder[2:]

      if "/" not in remainder:
        remainder = remainder + "/"
      self.host, remainder = remainder.split("/", 1)
      self.path = "/" + remainder

  def request(self, headers=None):
    if self.scheme == "data":
      return self.data

    if self.scheme == "file":
      file_path = self.path
      with open(file_path, "r") as f:
        return f.read()

    s = socket.socket(
        family=socket.AF_INET,
        type=socket.SOCK_STREAM,
        proto=socket.IPPROTO_TCP
    )
    if self.scheme == "http":
      self.port = 80
    elif self.scheme == "https":
      self.port = 443
    if self.scheme == "https":
      ctx = ssl.create_default_context()
      s = ctx.wrap_socket(s, server_hostname=self.host)
    if ":" in self.host:
      self.host, port = self.host.split(":", 1)
      self.port = int(port)
    s.connect((self.host, self.port))

    default_headers = {
        "Host": self.host,
        "Connection": "close",
        "User-Agent": "Shenanigan"
    }
    if headers:
        default_headers.update(headers)
    request_lines = [f"GET {self.path} HTTP/1.1\r\n"]
    for header_name, header_value in default_headers.items():
        request_lines.append(f"{header_name}: {header_value}\r\n")
    request_lines.append("\r\n")

    request_data = "".join(request_lines).encode("utf8")
    s.send(request_data)

    response = s.makefile("r", encoding="utf8", newline="\r\n")
    statusline = response.readline()
    version, status, explanation = statusline.split(" ", 2)
    response_headers = {}
    while True:
      line = response.readline()
      if line == "\r\n": break
      header, value = line.split(":", 1)
      response_headers[header.casefold()] = value.strip()
    assert "transfer-encoding" not in response_headers
    assert "content-encoding" not in response_headers
    body = response.read()
    s.close()
    return body

def show(body):
  in_tag = False
  in_html_entity = False
  html_entity = ""
  for c in body:
    if c == "<":
      in_tag = True
    elif c == ">":
      in_tag = False
    elif c == "&":
      in_html_entity = True
    elif in_html_entity:
      if c == ";":
        in_html_entity = False
        if html_entity == "lt":
          print("<", end="")
        elif html_entity == "gt":
          print(">", end="")
          # Add more cases for other HTML entities as needed
        else:
          print(f"&{html_entity};", end="")
        html_entity = ""
      else:
        html_entity += c
    elif not in_tag:
      print(c, end="")

def load(url, headers=None):
  body = url.request(headers=headers)
  show(body)

# Test with data scheme (using standard data: format)
load(URL("data:text/html,&lt;h1&gt;Hello from Data Scheme!&lt;/h1&gt;&lt;p&gt;This is a test.&lt;/p&gt;"))

<h1>Hello from Data Scheme!</h1><p>This is a test.</p>

#### 1-5 view-source 스킴

In [29]:
import socket
import ssl
import pathlib

class URL:
  def __init__(self, url):
    # split by the first ':' to get the scheme
    self.scheme, remainder = url.split(":", 1)
    assert self.scheme in ["http", "https", "file", "data", "view-source"]

    self.host = None
    self.path = None
    self.data = None

    if self.scheme == "view-source":
      self.inner_url = URL(remainder)
    elif self.scheme == "data":
      if "," not in remainder:
          raise ValueError(f"Invalid data URL: {url} - missing comma.")
      _, data_content = remainder.split(",", 1)
      self.data = data_content
    else:
      if remainder.startswith("//"):
        remainder = remainder[2:]
      if "/" not in remainder:
        remainder = remainder + "/"
      self.host, remainder = remainder.split("/", 1)
      self.path = "/" + remainder

  def request(self, headers=None):
    if self.scheme == "view-source":
      return self.inner_url.request(headers)

    if self.scheme == "data":
      return self.data

    if self.scheme == "file":
      file_path = self.path
      with open(file_path, "r") as f:
        return f.read()

    s = socket.socket(
        family=socket.AF_INET,
        type=socket.SOCK_STREAM,
        proto=socket.IPPROTO_TCP
    )
    if self.scheme == "http":
      self.port = 80
    elif self.scheme == "https":
      self.port = 443
    if self.scheme == "https":
      ctx = ssl.create_default_context()
      s = ctx.wrap_socket(s, server_hostname=self.host)
    if ":" in self.host:
      self.host, port = self.host.split(":", 1)
      self.port = int(port)
    s.connect((self.host, self.port))

    default_headers = {
        "Host": self.host,
        "Connection": "close",
        "User-Agent": "Shenanigan"
    }
    if headers:
        default_headers.update(headers)
    request_lines = [f"GET {self.path} HTTP/1.1\r\n"]
    for header_name, header_value in default_headers.items():
        request_lines.append(f"{header_name}: {header_value}\r\n")
    request_lines.append("\r\n")

    request_data = "".join(request_lines).encode("utf8")
    s.send(request_data)

    response = s.makefile("r", encoding="utf8", newline="\r\n")
    statusline = response.readline()
    version, status, explanation = statusline.split(" ", 2)
    response_headers = {}
    while True:
      line = response.readline()
      if line == "\r\n": break
      header, value = line.split(":", 1)
      response_headers[header.casefold()] = value.strip()
    assert "transfer-encoding" not in response_headers
    assert "content-encoding" not in response_headers
    body = response.read()
    s.close()
    return body

def show(body, scheme):
  if scheme == "view-source":
    print(body)
    return

  in_tag = False
  in_html_entity = False
  html_entity = ""
  for c in body:
    if c == "<":
      in_tag = True
    elif c == ">":
      in_tag = False
    elif c == "&":
      in_html_entity = True
    elif in_html_entity:
      if c == ";":
        in_html_entity = False
        if html_entity == "lt":
          print("<", end="")
        elif html_entity == "gt":
          print(">", end="")
        else:
          print(f"&{html_entity};", end="")
        html_entity = ""
      else:
        html_entity += c
    elif not in_tag:
      print(c, end="")

def load(url, headers=None):
  body = url.request(headers=headers)
  show(body, url.scheme)

load(URL("view-source:https://google.com/"))

<HTML><HEAD><meta http-equiv="content-type" content="text/html;charset=utf-8">
<TITLE>301 Moved</TITLE></HEAD><BODY>
<H1>301 Moved</H1>
The document has moved
<A HREF="https://www.google.com/">here</A>.
</BODY></HTML>



#### 1-6 Keep-alive

In [30]:
import socket
import ssl

class URL:
  SOCKET_CACHE = {}

  def __init__(self, url):
    self.scheme, url = url.split("://", 1)
    assert self.scheme in ["http", "https"]
    if "/" not in url:
      url = url + "/"
    self.host, url = url.split("/", 1)
    self.path = "/" + url
    self.port = 80 if self.scheme == "http" else 443
    if ":" in self.host:
      self.host, port = self.host.split(":", 1)
      self.port = int(port)

  def request(self, headers=None):
    connection_key = (self.scheme, self.host, self.port)

    # Reuse socket if available
    if connection_key in self.SOCKET_CACHE:
      s = self.SOCKET_CACHE[connection_key]
    else:
      s = socket.socket(
          family=socket.AF_INET,
          type=socket.SOCK_STREAM,
          proto=socket.IPPROTO_TCP
      )
      s.connect((self.host, self.port))
      if self.scheme == "https":
        ctx = ssl.create_default_context()
        s = ctx.wrap_socket(s, server_hostname=self.host)
      self.SOCKET_CACHE[connection_key] = s

    default_headers = {
        "Host": self.host,
        "Connection": "keep-alive",
        "User-Agent": "Shenanigan"
    }
    if headers:
        default_headers.update(headers)

    request_lines = [f"GET {self.path} HTTP/1.1\r\n"]
    for header_name, header_value in default_headers.items():
        request_lines.append(f"{header_name}: {header_value}\r\n")
    request_lines.append("\r\n")

    s.send("".join(request_lines).encode("utf8"))

    response = s.makefile("r", encoding="utf8", newline="\r\n")
    statusline = response.readline()
    version, status, explanation = statusline.split(" ", 2)

    response_headers = {}
    while True:
      line = response.readline()
      if line == "\r\n": break
      header, value = line.split(":", 1)
      response_headers[header.casefold()] = value.strip()

    # Must use Content-Length to read the body and keep the socket open
    content_length = int(response_headers.get("content-length", 0))
    body = response.read(content_length)

    # If the server explicitly says to close, remove from cache and close
    if response_headers.get("connection") == "close":
      s.close()
      del self.SOCKET_CACHE[connection_key]

    return body

def show(body):
  in_tag = False
  for c in body:
    if c == "<":
      in_tag = True
    elif c == ">":
      in_tag = False
    elif not in_tag:
      print(c, end="")

def load(url, headers=None):
  body = url.request(headers=headers)
  show(body)

# First request: Creates connection
print("--- First Request ---")
load(URL("http://httpbin.org/get"), {"Latte": "is horse."})

# Second request: Reuses the existing connection
print("\n--- Second Request (Reusing Socket) ---")
load(URL("http://httpbin.org/ip"))

--- First Request ---
{
  "args": {}, 
  "headers": {
    "Host": "httpbin.org", 
    "Latte": "is horse.", 
    "User-Agent": "Shenanigan", 
    "X-Amzn-Trace-Id": "Root=1-6a54cd61-62223e0c2b62c6562ec97fa7"
  }, 
  "origin": "34.81.86.44", 
  "url": "http://httpbin.org/get"
}

--- Second Request (Reusing Socket) ---
{
  "origin": "34.81.86.44"
}


#### 1-7 리다이렉트

In [43]:
import socket
import ssl

class URL:
  SOCKET_CACHE = {}

  def __init__(self, url):
    self.scheme, url = url.split("://", 1)
    assert self.scheme in ["http", "https"]
    if "/" not in url:
      url = url + "/"
    self.host, url = url.split("/", 1)
    self.path = "/" + url
    self.port = 80 if self.scheme == "http" else 443
    if ":" in self.host:
      self.host, port = self.host.split(":", 1)
      self.port = int(port)

  def request(self, headers=None):
    connection_key = (self.scheme, self.host, self.port)

    # Reuse socket if available
    if connection_key in self.SOCKET_CACHE:
      s = self.SOCKET_CACHE[connection_key]
    else:
      s = socket.socket(
          family=socket.AF_INET,
          type=socket.SOCK_STREAM,
          proto=socket.IPPROTO_TCP
      )
      s.connect((self.host, self.port))
      if self.scheme == "https":
        ctx = ssl.create_default_context()
        s = ctx.wrap_socket(s, server_hostname=self.host)
      self.SOCKET_CACHE[connection_key] = s

    default_headers = {
        "Host": self.host,
        "Connection": "keep-alive",
        "User-Agent": "Shenanigan"
    }
    if headers:
        default_headers.update(headers)

    request_lines = [f"GET {self.path} HTTP/1.1\r\n"]
    for header_name, header_value in default_headers.items():
        request_lines.append(f"{header_name}: {header_value}\r\n")
    request_lines.append("\r\n")

    s.send("".join(request_lines).encode("utf8"))

    response = s.makefile("r", encoding="utf8", newline="\r\n")
    statusline = response.readline()
    version, status, explanation = statusline.split(" ", 2)

    response_headers = {}
    while True:
      line = response.readline()
      if line == "\r\n": break
      header, value = line.split(":", 1)
      response_headers[header.casefold()] = value.strip()

    # Must use Content-Length to read the body and keep the socket open
    content_length = int(response_headers.get("content-length", 0))
    body = response.read(content_length)

    # If the server explicitly says to close, remove from cache and close
    if response_headers.get("connection") == "close":
      s.close()
      del self.SOCKET_CACHE[connection_key]

    return (status, response_headers, body)

def show(body):
  in_tag = False
  for c in body:
    if c == "<":
      in_tag = True
    elif c == ">":
      in_tag = False
    elif not in_tag:
      print(c, end="")

def load(url, headers=None, max_redirects=10):
  if max_redirects < 0:
    raise Exception("Too many redirects")

  status, response_headers, body = url.request(headers=headers)
  print(status, response_headers)
  if status.startswith("3") and "location" in response_headers:
      location = response_headers["location"]
      if location.startswith("/"):
          location = f"{url.scheme}://{url.host}{location}"
      elif "://" not in location:
          location = f"{url.scheme}://{url.host}/{location}"
      print(f"--- Redirecting to {location} ({max_redirects} left) ---")
      return load(URL(location), headers, max_redirects - 1)

  show(body)

load(URL("http://browser.engineering/redirect"))
load(URL("http://browser.engineering/redirect2"))
load(URL("http://browser.engineering/redirect3"))

--- Redirecting to http://browser.engineering/http.html (10 left) ---


KeyboardInterrupt: 

#### 1-8 캐싱


#### 1-9 압축

In [50]:
import socket
import ssl
import pathlib

class URL:
  def __init__(self, url):
    # split by the first ':' to get the scheme
    self.scheme, remainder = url.split(":", 1)
    assert self.scheme in ["http", "https", "file", "data", "view-source"]

    self.host = None
    self.path = None
    self.data = None

    if self.scheme == "view-source":
      self.inner_url = URL(remainder)
    elif self.scheme == "data":
      if "," not in remainder:
          raise ValueError(f"Invalid data URL: {url} - missing comma.")
      _, data_content = remainder.split(",", 1)
      self.data = data_content
    else:
      if remainder.startswith("//"):
        remainder = remainder[2:]
      if "/" not in remainder:
        remainder = remainder + "/"
      self.host, remainder = remainder.split("/", 1)
      self.path = "/" + remainder

  def request(self, headers=None):
    if self.scheme == "view-source":
      return self.inner_url.request(headers)

    if self.scheme == "data":
      return self.data

    if self.scheme == "file":
      file_path = self.path
      with open(file_path, "r") as f:
        return f.read()

    s = socket.socket(
        family=socket.AF_INET,
        type=socket.SOCK_STREAM,
        proto=socket.IPPROTO_TCP
    )
    if self.scheme == "http":
      self.port = 80
    elif self.scheme == "https":
      self.port = 443
    if self.scheme == "https":
      ctx = ssl.create_default_context()
      s = ctx.wrap_socket(s, server_hostname=self.host)
    if ":" in self.host:
      self.host, port = self.host.split(":", 1)
      self.port = int(port)
    s.connect((self.host, self.port))

    default_headers = {
        "Host": self.host,
        "Connection": "close",
        "Accept-Encoding": "gzip",
        "User-Agent": "Shenanigan"
    }
    if headers:
        default_headers.update(headers)
    request_lines = [f"GET {self.path} HTTP/1.1\r\n"]
    for header_name, header_value in default_headers.items():
        request_lines.append(f"{header_name}: {header_value}\r\n")
    request_lines.append("\r\n")

    request_data = "".join(request_lines).encode("utf8")
    s.send(request_data)

    response = s.makefile("r", encoding="utf8", newline="\r\n")
    statusline = response.readline()
    version, status, explanation = statusline.split(" ", 2)
    response_headers = {}
    while True:
      line = response.readline()
      if line == "\r\n": break
      header, value = line.split(":", 1)
      response_headers[header.casefold()] = value.strip()
    assert "transfer-encoding" not in response_headers
    body = response.read()
    if "content-encoding" in response_headers:
      if response_headers["content-encoding"] == "gzip":
        import gzip
        body = gzip.decompress(body)
    s.close()
    return (status, response_headers, body)

def show(body, scheme):
  if scheme == "view-source":
    print(body)
    return

  in_tag = False
  in_html_entity = False
  html_entity = ""
  for c in body:
    if c == "<":
      in_tag = True
    elif c == ">":
      in_tag = False
    elif c == "&":
      in_html_entity = True
    elif in_html_entity:
      if c == ";":
        in_html_entity = False
        if html_entity == "lt":
          print("<", end="")
        elif html_entity == "gt":
          print(">", end="")
        else:
          print(f"&{html_entity};", end="")
        html_entity = ""
      else:
        html_entity += c
    elif not in_tag:
      print(c, end="")

def load(url, headers=None):
  status, response_headers, body = url.request(headers=headers)
  print(status, response_headers)
  show(body, url.scheme)

load(URL("https://browser.engineering/"))

UnicodeDecodeError: 'utf-8' codec can't decode byte 0x8b in position 260: invalid start byte